In [1]:
# In Python
from pyspark.sql import SparkSession
import shutil
import os
# Create a SparkSession
spark = (SparkSession
  .builder
  .appName("SparkSQLExampleApp")
  .getOrCreate())

In [2]:
#create a database called learn_spark_db and tell Spark we want to use that database
spark.sql("CREATE DATABASE learn_spark_db")
spark.sql("USE learn_spark_db")

DataFrame[]

In [3]:
# Path to our US flight delays CSV file 
csv_file = "departuredelays.csv"
# Schema as defined in the preceding example
schema = "date STRING, delay INT, distance INT, origin STRING, destination STRING"

# Read CSV file into DataFrame
flights_df = spark.read.csv(csv_file, schema=schema)

# Даем имя таблице
table_name = "managed_us_delay_flights_tbl"

# Путь к директории таблицы в spark-warehouse
table_location = f"file:/home/jovyan/DE-101/Module7/DE%20-%20101%20Lab%207.5/examples/spark-warehouse/learn_spark_db.db/{table_name}"

# Удаляем таблицу из каталога Spark
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Удаляем физическую директорию, если она существует
try:
    shutil.rmtree(table_location.replace("file:", ""))
except FileNotFoundError:
    pass  # Если директория не существует, ничего не делаем
except Exception as e:
    print(f"Ошибка при удалении директории: {e}")

# Save the DataFrame as a managed table with overwrite mode
flights_df.write.mode("overwrite").saveAsTable(f"{table_name}")

In [4]:
df = spark.read.csv("departuredelays.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- date: integer (nullable = true)
 |-- delay: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)

+-------+-----+--------+------+-----------+
|   date|delay|distance|origin|destination|
+-------+-----+--------+------+-----------+
|1011245|    6|     602|   ABE|        ATL|
|1020600|   -8|     369|   ABE|        DTW|
|1021245|   -2|     602|   ABE|        ATL|
|1020605|   -4|     602|   ABE|        ATL|
|1031245|   -4|     602|   ABE|        ATL|
+-------+-----+--------+------+-----------+
only showing top 5 rows



In [5]:
#Creating an unmanaged table
# option 2
# Даем имя таблице
table_name = "us_delay_flights_tbl"

# Путь к директории таблицы в spark-warehouse
table_location = f"file:/home/jovyan/DE-101/Module7/DE%20-%20101%20Lab%207.5/examples/spark-warehouse/learn_spark_db.db/tmp/data/{table_name}"

# Удаляем таблицу из каталога Spark
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Удаляем физическую директорию, если она существует
try:
    shutil.rmtree(table_location.replace("file:", ""))
except FileNotFoundError:
    pass  # Если директория не существует, ничего не делаем
except Exception as e:
    print(f"Ошибка при удалении директории: {e}")

(flights_df
  .write
  .option("path", f"tmp/data/{table_name}")
  .saveAsTable("us_delay_flights_tbl"))

spark.sql("select * from us_delay_flights_tbl").show(5)

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01201755|    0|     449|   ORF|        ATL|
|01201610|   52|     449|   ORF|        ATL|
|01201441|    0|     449|   ORF|        ATL|
|01211755|  -15|     449|   ORF|        ATL|
|01210941|   -5|     449|   ORF|        ATL|
+--------+-----+--------+------+-----------+
only showing top 5 rows



In [6]:
#Creating Views
# In Python
df_sfo = spark.sql("SELECT date, delay, origin, destination FROM us_delay_flights_tbl WHERE origin = 'SFO'")
df_jfk = spark.sql("SELECT date, delay, origin, destination FROM us_delay_flights_tbl WHERE origin = 'JFK'")
# Create a temporary and global temporary view
spark.catalog.dropGlobalTempView("us_origin_airport_SFO_global_tmp_view")
df_sfo.createOrReplaceGlobalTempView("us_origin_airport_SFO_global_tmp_view")
spark.catalog.dropTempView("us_origin_airport_JFK_tmp_view")
df_jfk.createOrReplaceTempView("us_origin_airport_JFK_tmp_view")

In [7]:
# Проверяем объекты в learn_spark_db
spark.catalog.listTables('learn_spark_db')

[Table(name='managed_us_delay_flights_tbl', catalog='spark_catalog', namespace=['learn_spark_db'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='us_delay_flights_tbl', catalog='spark_catalog', namespace=['learn_spark_db'], description=None, tableType='EXTERNAL', isTemporary=False),
 Table(name='us_origin_airport_JFK_tmp_view', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [8]:
# Проверяем объекты в global_temp
spark.catalog.listTables('global_temp')

[Table(name='us_origin_airport_SFO_global_tmp_view', catalog=None, namespace=['global_temp'], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='us_origin_airport_JFK_tmp_view', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [9]:
# Обращаемся к local TempView
spark.sql("select * from us_origin_airport_JFK_tmp_view").show(5)
# Обращаемся к Global TempView
spark.sql("select * from global_temp.us_origin_airport_SFO_global_tmp_view").show(5)

+--------+-----+------+-----------+
|    date|delay|origin|destination|
+--------+-----+------+-----------+
|02010900|   -1|   JFK|        LAX|
|02011200|   -5|   JFK|        LAX|
|02011030|   -6|   JFK|        LAX|
|02011900|   -1|   JFK|        LAX|
|02011700|   -3|   JFK|        LAS|
+--------+-----+------+-----------+
only showing top 5 rows

+--------+-----+------+-----------+
|    date|delay|origin|destination|
+--------+-----+------+-----------+
|01011250|   55|   SFO|        JFK|
|01012230|    0|   SFO|        JFK|
|01010705|   -7|   SFO|        JFK|
|01010620|   -3|   SFO|        MIA|
|01010915|   -3|   SFO|        LAX|
+--------+-----+------+-----------+
only showing top 5 rows

